*[Cell 1]*

# Chapter 9: Streaming Medallion Architecture, Silver layer

**Databricks Free Edition version.** Run `medallion_bronze.ipynb` first: it fills the bronze table at `/Volumes/workspace/dldg/work/ch09/delta/bronze_raw/`. Apart from the paths, the code is the book's. The last section runs the bronze-to-silver pipeline as a real stream into the managed table `workspace.dldg.silver_ecomm_clickstream`.

In [ ]:
# [Cell 2]
# Databricks Free Edition: Spark runs on serverless via Databricks Connect (see dldg.py at the repo root)
from dldg import spark, WORK, rm
CH = f"{WORK}/ch09"  # replaces /opt/spark/work-dir/data

from delta.tables import DeltaTable
from pyspark.sql import DataFrame
from pyspark.sql.types import StructType, StructField, DateType, TimestampType, StringType, LongType, IntegerType, FloatType
from pyspark.sql.functions import col, when, lit, from_json, count, sum, min, max, avg, desc

In [ ]:
# [Cell 3]
# example 11-5
bronze_table_path = f"{CH}/delta/bronze_raw/"
dt_bronze_source: DeltaTable = DeltaTable.forPath(spark, bronze_table_path)

In [ ]:
# [Cell 4]
bronze_df = dt_bronze_source.toDF()
bronze_df.limit(1).show()

In [ ]:
# [Cell 5]
bronze_df.printSchema()

*[Cell 6]*

## Testing Transformations in Batch
> Tip: Before we dive into full end-to-end streaming, we can attack the problem in batch.

Once things are working as expected. We can consolidate the transformation steps and logic into a `transform` method with the signature `def transform_[some_qualifier](df: DataFrame) -> DataFrame:`. This lets us easily apply specific transformations in our pipelines that are a) testable, and b) easy to follow if looking directly at the pipeline.

~~~python
medallion_stream = (
    delta_source.readStream
    .options(**reader_options)
    .format("delta")
    .load()
    .transform(transform_from_json)
    .transform(transform_for_silver)
    .writeStream
    .format("delta")
    .options(**writer_options))

streaming_query = medallion_stream.toTable(f"{managed_table_name}")

spark.streams.awaitAnyTermination()
    
~~~

In [ ]:
# [Cell 7]
## Supply the ECommerce Schema for Permissive Passthrough
# from example 11-4

"""
public static final RowType ECOMMERCE_ROW_TYPE = new RowType(
          Arrays.asList(
                  new RowType.RowField("event_time", new VarCharType(VarCharType.MAX_LENGTH)),
                  new RowType.RowField("event_type", new VarCharType(VarCharType.MAX_LENGTH)),
                  new RowType.RowField("product_id", new IntType()),
                  new RowType.RowField("category_id", new BigIntType()),
                  new RowType.RowField("category_code", new VarCharType(VarCharType.MAX_LENGTH)),
                  new RowType.RowField("brand", new VarCharType(VarCharType.MAX_LENGTH)),
                  new RowType.RowField("price", new FloatType()),
                  new RowType.RowField("user_id", new IntType()),
                  new RowType.RowField("user_session", new VarCharType(VarCharType.MAX_LENGTH))
          ));
"""

known_schema: StructType = (
    StructType()
    .add(StructField("event_time", StringType()))
    .add(StructField("event_type", StringType()))
    .add(StructField("product_id", IntegerType()))
    .add(StructField("category_id", LongType()))
    .add(StructField("category_code", StringType()))
    .add(StructField("brand", StringType()))
    .add(StructField("price", FloatType()))
    .add(StructField("user_id", IntegerType()))
    .add(StructField("user_session", StringType()))
    .add(StructField("_corrupt", StringType(), True, {
    'comment': 'invalid rows go into _corrupt rather than simply being dropped'
    }))
)

def transform_from_json(input_df: DataFrame) -> DataFrame:
    return (
        input_df.withColumn(
            "ecomm", from_json(
                col("value").cast(StringType()),
                known_schema,
                options={
                    'mode': 'PERMISSIVE',
                    'columnNameOfCorruptRecord': '_corrupt'
                }
            )))

transformed_df = bronze_df.transform(transform_from_json)

In [ ]:
# [Cell 8]
transformed_df.show()

In [ ]:
# [Cell 9]
def transform_for_silver(input_df: DataFrame) -> DataFrame:
    return (
        input_df
        .select(
            col("event_date").alias("ingest_date"),
            col("timestamp").alias("ingest_timestamp"),
            col("ecomm.*")
        )
        .where(col("_corrupt").isNull())
        .drop("_corrupt")
    )

for_silver_df = (
    bronze_df
    .transform(transform_from_json)
    .transform(transform_for_silver))

In [ ]:
# [Cell 10]
for_silver_df.show()

In [ ]:
# [Cell 11]
# example 11-7
topN = (
    for_silver_df
    .groupBy("ingest_date", "category_id")
    .agg(
        count(col("product_id")).alias("impressions"),
        min(col("price")).alias("min_price"),
        max(col("price")).alias("max_price"),
        avg(col("price")).alias("avg_price")
    )
    .orderBy(desc("impressions"))
    .limit(5)
)

In [ ]:
# [Cell 12]
topN.show()

In [ ]:
# [Cell 13]
# Rows that failed to parse land in _corrupt instead of being silently dropped
transformed_df.where(col("ecomm._corrupt").isNotNull()).select("ecomm._corrupt").show(truncate=False)

*[Cell 14]*

## End to end: stream bronze into silver
The pipeline sketched in the markdown above, running for real. `availableNow` processes everything new in bronze and stops (serverless doesn't allow continuous triggers). Rerun `medallion_bronze.ipynb`'s "produce more" cell and then this cell, and only the new bronze rows flow through.

In [ ]:
# [Cell 15]
managed_table_name = "workspace.dldg.silver_ecomm_clickstream"
reader_options = {"ignoreDeletes": "true"}
writer_options = {"checkpointLocation": f"{CH}/_checkpoints/silver", "mergeSchema": "false"}

# Clean slate, so the stream starts from bronze version 0
spark.sql(f"DROP TABLE IF EXISTS {managed_table_name}")
rm(writer_options["checkpointLocation"])

medallion_stream = (
    spark.readStream
    .options(**reader_options)
    .format("delta")
    .load(bronze_table_path)
    .transform(transform_from_json)
    .transform(transform_for_silver)
    .writeStream
    .format("delta")
    .options(**writer_options)
    .trigger(availableNow=True))

streaming_query = medallion_stream.toTable(managed_table_name)
streaming_query.awaitTermination()  # instead of spark.streams.awaitAnyTermination()
[(p.batchId, p.numInputRows) for p in streaming_query.recentProgress]

In [ ]:
# [Cell 16]
spark.sql(f"""
  SELECT ingest_date, event_type, COUNT(*) AS events, ROUND(SUM(price), 2) AS revenue
  FROM {managed_table_name}
  GROUP BY ALL ORDER BY events DESC""").show()